## imports

In [4]:
import re

import pandas as pd
from IPython.display import display
from pathlib import Path
import os
from datetime import datetime, timedelta
import numpy as np
import urllib3
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import requests

## 1. API para recolher os dados

In [9]:
url = "https://euromillions.api.pedromealha.dev/v1/draws"  # conforme documentação atualizada
response = requests.get(url, timeout=10)
data = response.json()  # estrutura JSON com todos os sorteios

# Exemplificando conversão a DataFrame
df = pd.json_normalize(data)
df.head()

,date,draw_id,has_winner,id,numbers,prizes,stars
0,2004-02-13,12004,True,1,"[16, 29, 32, 36, 41]","[{'matched_numbers': 2, 'matched_stars': 1, 'p...","[7, 9]"
1,2004-02-20,22004,False,2,"[7, 13, 39, 47, 50]","[{'matched_numbers': 2, 'matched_stars': 1, 'p...","[2, 5]"
2,2004-02-27,32004,False,3,"[14, 18, 19, 31, 37]","[{'matched_numbers': 2, 'matched_stars': 1, 'p...","[4, 5]"
3,2004-03-05,42004,True,4,"[4, 7, 33, 37, 39]","[{'matched_numbers': 2, 'matched_stars': 1, 'p...","[1, 5]"
4,2004-03-12,52004,False,5,"[15, 24, 28, 44, 47]","[{'matched_numbers': 2, 'matched_stars': 1, 'p...","[4, 5]"


In [11]:
if "draw_id" in df.columns and "id" in df.columns:
    del df["draw_id"]
    del df["id"]

# Criar novas colunas a partir da lista
numbers_expanded = pd.DataFrame(df["numbers"].tolist(), 
                                columns=[f"N{i+1}" for i in range(len(df["numbers"][0]))])

stars_expanded = pd.DataFrame(df["stars"].tolist(), 
                                columns=[f"E{i+1}" for i in range(len(df["stars"][0]))])

# Concatenar ao DataFrame original
df_final = pd.concat([df.drop(columns=["numbers"]), numbers_expanded], axis=1)
df_final = pd.concat([df_final.drop(columns=["stars"]), stars_expanded], axis=1)

# Alterar o tipo de dados 
df_final["has_winner"] = df_final["has_winner"].astype(int)

df_final.head()

,date,has_winner,prizes,N1,N2,N3,N4,N5,E1,E2
0,2004-02-13,1,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",16,29,32,36,41,7,9
1,2004-02-20,0,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",7,13,39,47,50,2,5
2,2004-02-27,0,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",14,18,19,31,37,4,5
3,2004-03-05,1,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",4,7,33,37,39,1,5
4,2004-03-12,0,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",15,24,28,44,47,4,5


In [13]:
df_final

,date,has_winner,prizes,N1,N2,N3,N4,N5,E1,E2
0,2004-02-13,1,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",16,29,32,36,41,7,9
1,2004-02-20,0,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",7,13,39,47,50,2,5
2,2004-02-27,0,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",14,18,19,31,37,4,5
3,2004-03-05,1,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",4,7,33,37,39,1,5
4,2004-03-12,0,"[{'matched_numbers': 2, 'matched_stars': 1, 'p...",15,24,28,44,47,4,5
...,...,...,...,...,...,...,...,...,...,...
1863,2025-08-01,0,"[{'matched_numbers': 2, 'matched_stars': 0, 'p...",4,16,25,29,30,2,10
1864,2025-08-05,0,"[{'matched_numbers': 2, 'matched_stars': 0, 'p...",1,3,5,42,47,5,10
1865,2025-08-08,0,"[{'matched_numbers': 2, 'matched_stars': 0, 'p...",2,12,19,34,44,6,10
1866,2025-08-12,0,"[{'matched_numbers': 2, 'matched_stars': 0, 'p...",18,28,42,46,48,3,9


## 2. Guardar os dados em ficheiro CSV

In [ ]:
# 2. Guardar os dados num ficheiro csv
p = Path.cwd()
p = p.parent
data_folder = p / "data"
nome_ficheiro = "euromilhoesTeste.csv"

caminho_completo = os.path.join(data_folder, nome_ficheiro)
os.makedirs(data_folder, exist_ok=True)
df_final.to_csv(caminho_completo, index=False)

- - -

## Extra - WebScrapping (Problema de acessos)

In [6]:
def todas_tercas_e_sextas(desde_ano, ate_ano):
    if desde_ano > datetime.now().year or desde_ano < 2004:
        raise ValueError
    if desde_ano == 2004:
        data = datetime(desde_ano, 2, 13)
    else:
        data = datetime(desde_ano, 1, 1)

    # Cria uma lista para armazenar as datas
    datas_tercas_e_sextas = []

    # Itera sobre todos os dias desde desde_ano até a data final
    while data.year <= ate_ano and data <= datetime.now():
        if data.weekday() == 4 and data <= datetime(2011, 5, 6):
            datas_tercas_e_sextas.append((data.strftime("%d-%m-%Y")))
        # Adiciona a data à lista se for terça ou sexta-feira
        elif data > datetime(2011, 5, 6) and (
                data.weekday() == 1 or data.weekday() == 4):  # 0 = segunda, 1 = terça, ..., 6 = domingo
            datas_tercas_e_sextas.append(data.strftime("%d-%m-%Y"))

        # Avança para o próximo dia
        data += timedelta(days=1)

    return datas_tercas_e_sextas


# Define os anos de início e fim
desde_ano = 2004  # colocar 2004
ate_ano = datetime.now().year

# Obtém a lista de datas
datas = todas_tercas_e_sextas(desde_ano, ate_ano)

colunasTabela = [
    'Data',
    'DiaSemana',
    'N1',
    'N2',
    'N3',
    'N4',
    'N5',
    'E1',
    'E2',
]

euromilhoes = pd.DataFrame(columns=colunasTabela)  # Criação do DataFrame Final
euromilhoes

,Data,DiaSemana,N1,N2,N3,N4,N5,E1,E2


In [7]:
limitar = True

if limitar:
    datas = datas[:10]
datas

['13-02-2004',
 '20-02-2004',
 '27-02-2004',
 '05-03-2004',
 '12-03-2004',
 '19-03-2004',
 '26-03-2004',
 '02-04-2004',
 '09-04-2004',
 '16-04-2004']

In [8]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Cabeçalhos para simular um browser normal
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/115.0 Safari/537.36",
    "Accept-Language": "pt-PT,pt;q=0.9,en-US;q=0.8,en;q=0.7"
})

# Exemplo de colunas (ajusta se já tiveres definido)
colunasTabela = ["Data", "DiaSemana", "N1", "N2", "N3", "N4", "N5", "E1", "E2"]

def processar_data(data):
    """Função que faz scraping de uma data e devolve DataFrame de 1 linha"""
    try:
        url = f"https://www.euro-millions.com/results/{data}"
        print(f"URL: {url}")

        resp = session.get(url, timeout=10)

        if resp.status_code != 200:
            print(f"Erro HTTP {resp.status_code} em {data}")
            return None

        pagina = BeautifulSoup(resp.text, "html.parser")

        tabelaNumero = pagina.find_all("li", class_="resultBall ball")
        tabelaEstrela = pagina.find_all("li", class_="resultBall lucky-star")

        # Se não encontrar resultados, ignora
        if not tabelaNumero or not tabelaEstrela:
            print(f"Nenhum resultado encontrado em {data}")
            return None

        # Recolher dados
        dados = [data]
        dados.append(datetime.strptime(data, "%d-%m-%Y").strftime("%a"))  # abreviado (ex: Seg, Ter)

        # Adicionar números
        for i, celula in enumerate(tabelaNumero[:5]):
            dados.append(int(celula.text))

        # Adicionar estrelas
        for i, celula in enumerate(tabelaEstrela[:2]):
            dados.append(int(celula.text))

        # Converter para DataFrame de 1 linha
        matriz_np = np.array(dados).reshape(1, 9)
        df = pd.DataFrame(matriz_np, columns=colunasTabela)

        return df

    except Exception as e:
        print(f"Erro em {data}: {e}")
        return None

# Paralelização com barra de progresso
results = []
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(processar_data, d) for d in datas]
    for f in tqdm(as_completed(futures), total=len(futures)):
        results.append(f.result())

# Concatenar todos os DataFrames válidos
euromilhoes = pd.concat([df for df in results if df is not None], ignore_index=True)


URL: https://www.euro-millions.com/results/13-02-2004
URL: https://www.euro-millions.com/results/20-02-2004
URL: https://www.euro-millions.com/results/27-02-2004


  0%|          | 0/10 [00:00<?, ?it/s]

Erro HTTP 403 em 13-02-2004
URL: https://www.euro-millions.com/results/05-03-2004
Erro HTTP 403 em 27-02-2004
URL: https://www.euro-millions.com/results/12-03-2004


100%|██████████| 10/10 [00:00<00:00, 30.77it/s]

Erro HTTP 403 em 20-02-2004
URL: https://www.euro-millions.com/results/19-03-2004
Erro HTTP 403 em 12-03-2004
URL: https://www.euro-millions.com/results/26-03-2004
Erro HTTP 403 em 19-03-2004
URL: https://www.euro-millions.com/results/02-04-2004
Erro HTTP 403 em 05-03-2004
URL: https://www.euro-millions.com/results/09-04-2004
Erro HTTP 403 em 26-03-2004
URL: https://www.euro-millions.com/results/16-04-2004
Erro HTTP 403 em 02-04-2004
Erro HTTP 403 em 09-04-2004
Erro HTTP 403 em 16-04-2004


ValueError: No objects to concatenate